In [1]:
import pandas as pd,numpy as np
import matplotlib.pyplot as plt,seaborn as sns
import kagglehub

In [2]:
path = kagglehub.dataset_download("yasserh/loan-default-dataset")

df = pd.read_csv(path + "/Loan_Default.csv", encoding="latin1")

df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [3]:
df = df.drop(['year','ID'],axis=1)
NumCols = df.select_dtypes(exclude=object).columns
CatCols = df.select_dtypes(object).columns

In [4]:
NumCols

Index(['loan_amount', 'rate_of_interest', 'Interest_rate_spread',
       'Upfront_charges', 'term', 'property_value', 'income', 'Credit_Score',
       'LTV', 'Status', 'dtir1'],
      dtype='object')

In [5]:
df.isnull().sum()

loan_limit                    3344
Gender                           0
approv_in_adv                  908
loan_type                        0
loan_purpose                   134
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest             36439
Interest_rate_spread         36639
Upfront_charges              39642
term                            41
Neg_ammortization              121
interest_only                    0
lump_sum_payment                 0
property_value               15098
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                        9150
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                            200
submission_of_application      200
LTV                          15098
Region              

In [6]:
from src.transformers.Transformers import EnCoder,CatGroupModeImputer,NumGroupMeanImputer
from sklearn.pipeline import Pipeline

In [7]:
NumGrpImputeCols= ['Region','loan_type','business_or_commercial']
CatGrpImputeCols = ['Status','loan_type','Security_Type','credit_type','Gender','co-applicant_credit_type']

In [8]:
# NumtargetimputeCols =['income','property_value','term']
# CattargetimputeCols=['loan_limit','approv_in_adv','loan_purpose','Neg_ammortization','submission_of_application','age']

In [9]:
Selective_num_imputer = Pipeline([
    ('income',NumGroupMeanImputer(group_cols=NumGrpImputeCols,target_col='income')),
    ('property_value',NumGroupMeanImputer(group_cols=NumGrpImputeCols,target_col='property_value')),
    ('term',NumGroupMeanImputer(group_cols=NumGrpImputeCols,target_col='term'))
])



Selective_cat_imputer = Pipeline([
    ('loan_limit',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='loan_limit')),
    ('approv_in_adv',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='approv_in_adv')),
    ('loan_purpose',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='loan_purpose')),
    ('Neg_ammortization',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='Neg_ammortization')),
    ('submission_of_application',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='submission_of_application')),
    ('age',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='age'))
])

In [10]:
SelectiveImputer = Pipeline([
    ('num',Selective_num_imputer),
    ('cat',Selective_cat_imputer)
])

In [11]:
newdf = SelectiveImputer.fit_transform(df,'y')
newdf.isnull().sum()

loan_limit                       0
Gender                           0
approv_in_adv                    0
loan_type                        0
loan_purpose                     0
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest             36439
Interest_rate_spread         36639
Upfront_charges              39642
term                             0
Neg_ammortization                0
interest_only                    0
lump_sum_payment                 0
property_value                   0
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                           0
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                              0
submission_of_application        0
LTV                          15098
Region              

In [12]:
newdf['LTV'] = newdf['LTV'].fillna(newdf['loan_amount'] / newdf['property_value'])

In [13]:
newdf.isnull().sum()

loan_limit                       0
Gender                           0
approv_in_adv                    0
loan_type                        0
loan_purpose                     0
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest             36439
Interest_rate_spread         36639
Upfront_charges              39642
term                             0
Neg_ammortization                0
interest_only                    0
lump_sum_payment                 0
property_value                   0
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                           0
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                              0
submission_of_application        0
LTV                              0
Region              

In [14]:
from sklearn.ensemble import RandomForestRegressor

In [15]:
for col in CatCols:
    print("*-"*30)
    print(newdf[col].value_counts())

*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
loan_limit
cf     138692
ncf      9978
Name: count, dtype: int64
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
Gender
Male                 42346
Joint                41399
Sex Not Available    37659
Female               27266
Name: count, dtype: int64
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
approv_in_adv
nopre    125529
pre       23141
Name: count, dtype: int64
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
loan_type
type1    113173
type2     20762
type3     14735
Name: count, dtype: int64
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
loan_purpose
p3    55987
p4    54871
p1    34538
p2     3274
Name: count, dtype: int64
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
Credit_Worthiness
l1    142344
l2      6326
Name: count, dtype: int64
*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-
open_credit
nopc    148114
opc        556
Name: count,

### ordinal cols, and why :
| Column              | Reason                            |
| ------------------- | --------------------------------- |
| `age`               | age groups have order             |
| `total_units`       | 1U < 2U < 3U < 4U                 |
| `credit_type`       | can be ordered by risk (optional) |


### ohe cols 
| Column                    |
| ------------------------- |
| Gender                    |
| loan_limit                |
| approv_in_adv             |
|Credit_Worthiness          |
| loan_type                 |
| loan_purpose              |
| open_credit               |
| business_or_commercial    |
| Neg_ammortization         |
| interest_only             |
| lump_sum_payment          |
| construction_type         |
| occupancy_type            |
| Secured_by                |
| submission_of_application |
| Region                    |
| Security_Type             |
| co-applicant_credit_type  |


In [16]:
ageLabelOrder  = ['<25','25-34', '35-44', '45-54','55-64', '65-74', '>74']
creditypeLabelOrder = [ 'CIB', 'CRIF' ,'EXP','EQUI']
totalunitLabelorder = ['1U','2U','3U','4U']

In [17]:
ordinalcols = ['age','total_units','credit_type']
ordinalCategories = [ageLabelOrder,totalunitLabelorder,creditypeLabelOrder]

In [18]:
ohecols = [
    "Gender",
    "loan_limit",
    "approv_in_adv",
    "Credit_Worthiness",
    "loan_type",
    "loan_purpose",
    "open_credit",
    "business_or_commercial",
    "Neg_ammortization",
    "interest_only",
    "lump_sum_payment",
    "construction_type",
    "occupancy_type",
    "Secured_by",
    "submission_of_application",
    "Region",
    "Security_Type",
    "co-applicant_credit_type"
]

In [19]:
encodePipe = Pipeline([
    ('encode',EnCoder(OrdinalCols=ordinalcols,OrdinalCategories=ordinalCategories,OheCols=ohecols))
])

In [20]:
encodePipe.fit(newdf)

Pipeline(steps=[('encode',
                 EnCoder(OheCols=['Gender', 'loan_limit', 'approv_in_adv',
                                  'Credit_Worthiness', 'loan_type',
                                  'loan_purpose', 'open_credit',
                                  'business_or_commercial', 'Neg_ammortization',
                                  'interest_only', 'lump_sum_payment',
                                  'construction_type', 'occupancy_type',
                                  'Secured_by', 'submission_of_application',
                                  'Region', 'Security_Type',
                                  'co-applicant_credit_type'],
                         OrdinalCategories=[['<25', '25-34', '35-44', '45-54',
                                             '55-64', '65-74', '>74'],
                                            ['1U', '2U', '3U', '4U'],
                                            ['CIB', 'CRIF', 'EXP', 'EQUI']],
                         OrdinalCols=['age', 'total_units', 'credit_type']))])

In [21]:
newdf_tsf = encodePipe.transform(newdf)

In [22]:
y = newdf[['rate_of_interest','Interest_rate_spread']]
X = newdf.drop(['rate_of_interest','Interest_rate_spread','Upfront_charges','dtir1'],axis=1)

In [23]:
y_roi = y['rate_of_interest']

mask_roi = y_roi.notna()

X_train_roi = X[mask_roi]
y_train_roi = y_roi[mask_roi]

X_test_roi = X[~mask_roi]

In [24]:
X_test_roi.shape

(36439, 28)

In [25]:
rf_pipe_roi = Pipeline(
    [
        ('selective_impute',SelectiveImputer),
        ('EncodeFeatures',encodePipe),
        ('rf',RandomForestRegressor())
    ]
)

In [26]:
rf_pipe_roi.fit(X_train_roi,y_train_roi)

Pipeline(steps=[('selective_impute',
                 Pipeline(steps=[('num',
                                  Pipeline(steps=[('income',
                                                   NumGroupMeanImputer(group_cols=['Region',
                                                                                   'loan_type',
                                                                                   'business_or_commercial'],
                                                                       target_col='income')),
                                                  ('property_value',
                                                   NumGroupMeanImputer(group_cols=['Region',
                                                                                   'loan_type',
                                                                                   'business_or_commercial'],
                                                                       target_col='property_value')),
                                                  ('term',
                                                   NumGroupMeanImputer(group_col...
                                                   'lump_sum_payment',
                                                   'construction_type',
                                                   'occupancy_type',
                                                   'Secured_by',
                                                   'submission_of_application',
                                                   'Region', 'Security_Type',
                                                   'co-applicant_credit_type'],
                                          OrdinalCategories=[['<25', '25-34',
                                                              '35-44', '45-54',
                                                              '55-64', '65-74',
                                                              '>74'],
                                                             ['1U', '2U', '3U',
                                                              '4U'],
                                                             ['CIB', 'CRIF',
                                                              'EXP', 'EQUI']],
                                          OrdinalCols=['age', 'total_units',
                                                       'credit_type']))])),
                ('rf', RandomForestRegressor())])

In [27]:
y_pred_roi = rf_pipe_roi.predict(X_test_roi)

D:\project 4\myenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [11, 13, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [28]:
X_test_roi.columns[11:17]

Index(['interest_only', 'lump_sum_payment', 'property_value',
       'construction_type', 'occupancy_type', 'Secured_by'],
      dtype='object')

In [29]:
y_pred_roi

array([4.3015 , 4.007  , 4.55695, ..., 3.81675, 3.76675, 3.8641 ])

In [30]:
mask = newdf["rate_of_interest"].isna()
newdf.loc[mask, "rate_of_interest"] = y_pred_roi

In [31]:
newdf.isna().sum()

loan_limit                       0
Gender                           0
approv_in_adv                    0
loan_type                        0
loan_purpose                     0
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest                 0
Interest_rate_spread         36639
Upfront_charges              39642
term                             0
Neg_ammortization                0
interest_only                    0
lump_sum_payment                 0
property_value                   0
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                           0
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                              0
submission_of_application        0
LTV                              0
Region              

In [32]:
newdf = newdf[newdf['rate_of_interest']!=0]

In [33]:
newdf.columns

Index(['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose',
       'Credit_Worthiness', 'open_credit', 'business_or_commercial',
       'loan_amount', 'rate_of_interest', 'Interest_rate_spread',
       'Upfront_charges', 'term', 'Neg_ammortization', 'interest_only',
       'lump_sum_payment', 'property_value', 'construction_type',
       'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type',
       'Credit_Score', 'co-applicant_credit_type', 'age',
       'submission_of_application', 'LTV', 'Region', 'Security_Type', 'Status',
       'dtir1'],
      dtype='object')

In [34]:
from src.transformers.Transformers import DtirImputer,HasFeaturesTransform

In [35]:
imputedtir = Pipeline([
    ('dtirimputer',DtirImputer())
])

In [36]:
newdf  = imputedtir.fit_transform(newdf)

In [37]:
newdf.isna().sum()

loan_limit                       0
Gender                           0
approv_in_adv                    0
loan_type                        0
loan_purpose                     0
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest                 0
Interest_rate_spread         36639
Upfront_charges              39641
term                             0
Neg_ammortization                0
interest_only                    0
lump_sum_payment                 0
property_value                   0
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                           0
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                              0
submission_of_application        0
LTV                              0
Region              

In [38]:
hasfeaturetransformer = HasFeaturesTransform(col='Upfront_charges',value=0,method='>')

In [39]:
newdf = hasfeaturetransformer.fit_transform(newdf)

In [40]:
newdf.isna().sum()

loan_limit                       0
Gender                           0
approv_in_adv                    0
loan_type                        0
loan_purpose                     0
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest                 0
Interest_rate_spread         36639
Upfront_charges              39641
term                             0
Neg_ammortization                0
interest_only                    0
lump_sum_payment                 0
property_value                   0
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                           0
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                              0
submission_of_application        0
LTV                              0
Region              

In [41]:
newdf['Upfront_charges'] = newdf['Upfront_charges'].fillna(0)

In [42]:
newdf.drop('Interest_rate_spread',axis=1,inplace=True)

In [43]:
newdf.dropna(axis=0,inplace=True)

In [44]:
newdf.Status.value_counts()

Status
0    112030
1     35727
Name: count, dtype: int64

In [45]:
newdf.to_csv('data/Cleaned_loan_records.csv',index=False)

In [46]:
newdf.isna().sum()

loan_limit                   0
Gender                       0
approv_in_adv                0
loan_type                    0
loan_purpose                 0
Credit_Worthiness            0
open_credit                  0
business_or_commercial       0
loan_amount                  0
rate_of_interest             0
Upfront_charges              0
term                         0
Neg_ammortization            0
interest_only                0
lump_sum_payment             0
property_value               0
construction_type            0
occupancy_type               0
Secured_by                   0
total_units                  0
income                       0
credit_type                  0
Credit_Score                 0
co-applicant_credit_type     0
age                          0
submission_of_application    0
LTV                          0
Region                       0
Security_Type                0
Status                       0
dtir1                        0
has_Upfront_charges          0
dtype: i